In [13]:
import pandas as pd
import os
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from openai import OpenAI
from pydantic import BaseModel
import json

# Load API key
with open("key.txt", "r") as f:
    api_key = f.read().strip()

client = OpenAI(api_key=api_key)


In [14]:
import re

# Arabic tashkeel unicode range
TASHKEEL_REGEX = re.compile(r'[\u0617-\u061A\u064B-\u0652]')

def remove_tashkeel(text):
    if not isinstance(text, str):
        return text
    return re.sub(TASHKEEL_REGEX, '', text)


In [15]:
# 2. Data Loading (MAJAZ ONLY)
# ============================
print("Loading Majaz dataset only...")
df_majaz = pd.read_csv("majaz_dataset.csv")
TEXT_COL = "البيت_الكامل"

# Clean
df_majaz = df_majaz[[TEXT_COL]].dropna()
df_majaz[TEXT_COL] = df_majaz[TEXT_COL].apply(remove_tashkeel)

# Select a sample to test (e.g., 50 random verses to save cost/time)
# OR comment out `.sample(50)` to run on the whole dataset.
X_test = df_majaz[TEXT_COL].sample(50, random_state=42).values 
y_test = [1] * len(X_test)  # All are True (1)

print(f"Testing on {len(X_test)} Majaz verses (Zero-Shot).")

Loading Majaz dataset only...
Testing on 50 Majaz verses (Zero-Shot).


In [16]:
# 3. Define Clean Zero-Shot Prompt
# ================================
class VerseClassification(BaseModel):
    id: int
    reasoning: str 
    is_majaz: int

class BatchResponse(BaseModel):
    results: list[VerseClassification]

SYSTEM_PROMPT = """
أنت ناقد خبير بلاغي ومتخصص في عنصر المجاز
سأعطيك بيت شعر، افحص اذا كان فيه العنصر البلاغي المجاز ام لا

المطلوب:
حلل البيت، ثم قرر: 1 (مجاز) أو 0 (ليس مجازاً).
"""

def get_predictions_batch(verses, start_index):
    formatted_verses = "\n".join([f"ID {i+start_index}: {v}" for i, v in enumerate(verses)])

    user_content = f"""
    الأبيات:
    {formatted_verses}
    """

    try:
        completion = client.beta.chat.completions.parse(
            model="gpt-5.2-2025-12-11", # Strongest available model for reasoning
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_content},
            ],
            response_format=BatchResponse,
            temperature=0.0,
        )
        return completion.choices[0].message.parsed.results
    except Exception as e:
        print(f"Error: {e}")
        return []

In [17]:
# 4. Execution
# ============
BATCH_SIZE = 1 
predictions_map = {}

print("Running Zero-Shot Test...")

for i in tqdm(range(0, len(X_test), BATCH_SIZE)):
    batch_verses = X_test[i : i + BATCH_SIZE]
    results = get_predictions_batch(batch_verses, start_index=i)
    
    for res in results:
        predictions_map[res.id] = {"pred": res.is_majaz, "reason": res.reasoning}

Running Zero-Shot Test...


100%|██████████| 50/50 [04:28<00:00,  5.38s/it]


In [18]:
# 5. Results
# ==========
final_preds = []
final_reasons = []

for i in range(len(X_test)):
    if i in predictions_map:
        final_preds.append(predictions_map[i]["pred"])
        final_reasons.append(predictions_map[i]["reason"])
    else:
        final_preds.append(0) # Default failure
        final_reasons.append("Error")

# Calculate Recall (Since all Ground Truth are 1)
# Accuracy here IS Recall because there are no negatives in the test set.
recall = accuracy_score(y_test, final_preds)

print(f"\nRecall (Sensitivity): {recall:.2%}")
print(f"Correctly identified {sum(final_preds)} out of {len(X_test)} Majaz verses.")

# Save False Negatives (Missed Majaz)
results_df = pd.DataFrame({
    "verse": X_test,
    "y_true": y_test,
    "y_pred": final_preds,
    "reasoning": final_reasons
})

# Filter to show what the model missed
missed_df = results_df[results_df["y_pred"] == 0]
print(f"\nMissed Cases (False Negatives): {len(missed_df)}")

if not missed_df.empty:
    print(missed_df[["verse", "reasoning"]].head())
    missed_df.to_csv("zero_shot_missed_majaz.csv", index=False, encoding="utf-8-sig")


Recall (Sensitivity): 86.00%
Correctly identified 43 out of 50 Majaz verses.

Missed Cases (False Negatives): 7
                                                verse  \
4   قال قرينه ربنا ما أطغيته ولٰكن كان في ضلال بعي...   
16  تجفل حينا، ثم تدنو كأنما تنادي طلا بآلواد أعجز...   
17  أعز بني آلدنيا وأعلى ذوي آلعلا، وأكرم من فوق آ...   
25  إذا اندفعت تمشي الهوينى كأنها قناة تعلت لينها ...   
29  نظرت إليها نظرة ما يسرني بها حمر أنعام البلاد ...   

                                            reasoning  
4   في قوله: «قال قرينه» إسنادُ القول إلى «القرين»...  
16  في البيت تصوير لحركة الدابة/الناقة: «تجفل حينا...  
17  البيت يقوم على أسلوب تفضيل ومدح مباشر: «أعز… و...  
25  في البيت تشبيه صريح بأداة التشبيه (كأنها) بين ...  
29  في البيت يقول: «نظرتُ إليها نظرةً ما يسرّني به...  
